# Stochastic Optimization

**Chapter 5**

In deterministic optimization models, all parameters are assumed to be known with certainty. In reality, many important inputs — such as customer demand, market prices, travel times, or machine yields — are uncertain. **Stochastic optimization** provides a principled framework for making good decisions when some parameters are random variables.

This chapter introduces **two-stage stochastic programming with recourse** and the practical **Sample Average Approximation (SAA)** technique. We use the classic newsvendor problem as our main example because it clearly illustrates the key ideas.

## 5.1 Two-Stage Stochastic Programming

In a two-stage stochastic program, decisions are split into:

- **First-stage decisions** (here-and-now): Chosen before uncertainty is revealed.
- **Second-stage decisions** (recourse): Adaptive actions taken after the random outcome is observed.

The goal is to minimize the first-stage cost plus the *expected* second-stage recourse cost.

## 5.2 The Stochastic Newsvendor Problem

A vendor must decide how many units to order **before** knowing actual daily demand. Ordering too many incurs an overage (holding) cost. Ordering too few incurs an underage (lost sales) cost.

We model demand as a random variable and generate discrete scenarios to approximate the expectation.

In [ ]:
import numpy as np
import pulp
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
np.random.seed(42)

# Problem parameters
c_o = 3.0           # Overage (holding) cost per unit
c_u = 12.0          # Underage (lost sales) cost per unit
purchase_cost = 8.0 # Cost to purchase each unit

# Generate demand scenarios
n_scenarios = 1000
mean_demand = 150
std_demand = 45

demand_scenarios = np.random.normal(mean_demand, std_demand, n_scenarios)
demand_scenarios = np.maximum(demand_scenarios, 20)   # Ensure minimum demand of 20

# Visualize demand distribution
plt.figure(figsize=(10, 6))
sns.histplot(demand_scenarios, bins=50, kde=True, color='steelblue', alpha=0.7)
plt.axvline(mean_demand, color='red', linestyle='--', linewidth=2, label=f'Mean Demand = {mean_demand}')
plt.title('Distribution of Stochastic Demand Scenarios')
plt.xlabel('Daily Demand (units)')
plt.ylabel('Frequency')
plt.legend()
plt.show()

print(f"Number of scenarios: {n_scenarios}")
print(f"Mean demand: {demand_scenarios.mean():.1f} units")
print(f"Standard deviation: {demand_scenarios.std():.1f} units")

## 5.3 Sample Average Approximation (SAA)

We approximate the expected cost by averaging over the generated scenarios. This turns the stochastic problem into a large deterministic linear program that can be solved with standard solvers.

In [ ]:
# SAA model
prob = pulp.LpProblem("Stochastic_Newsvendor_SAA", pulp.LpMinimize)

# First-stage decision variable
x = pulp.LpVariable("Order_Quantity", lowBound=0)

# Second-stage recourse variables
over = [pulp.LpVariable(f"over_{i}", lowBound=0) for i in range(n_scenarios)]
under = [pulp.LpVariable(f"under_{i}", lowBound=0) for i in range(n_scenarios)]

# Objective: purchase cost + expected recourse cost
prob += purchase_cost * x + (1.0 / n_scenarios) * pulp.lpSum(
    c_o * over[i] + c_u * under[i] for i in range(n_scenarios)
)

# Recourse constraints
for i in range(n_scenarios):
    prob += over[i] >= x - demand_scenarios[i]
    prob += under[i] >= demand_scenarios[i] - x

# Solve
prob.solve(pulp.PULP_CBC_CMD(msg=False))

print("=== SAA Optimal Solution ===")
print(f"Optimal order quantity : {pulp.value(x):.1f} units")
print(f"Expected total cost    : ${pulp.value(prob.objective):.2f}")

## 5.4 Critical Fractile Rule (Analytical Benchmark)

For the classic newsvendor problem, there is a well-known closed-form solution using the **critical fractile**:

$$ F(x^*) = \frac{c_u}{c_u + c_o} $$

where $F$ is the cumulative distribution function of demand.

In [ ]:
critical_ratio = c_u / (c_o + c_u)
analytical_order = np.quantile(demand_scenarios, critical_ratio)

print(f"Critical fractile (target service level): {critical_ratio:.3f}")
print(f"Analytical optimal order quantity       : {analytical_order:.1f} units")

saa_order = pulp.value(x)
print(f"\nDifference (SAA vs analytical): {abs(saa_order - analytical_order):.2f} units")

## 5.5 Two-Stage Capacity Planning Example

Here we apply the same ideas to a capacity planning problem where capacity must be decided before demand is known.

In [ ]:
# Two-stage capacity planning
prob2 = pulp.LpProblem("Two_Stage_Capacity_Planning", pulp.LpMinimize)

capacity = pulp.LpVariable("Built_Capacity", lowBound=0)

scenarios = [100, 140, 190, 240]
probs = [0.25, 0.30, 0.30, 0.15]

expected_recourse = 0

for d, p in zip(scenarios, probs):
    extra = pulp.LpVariable(f"extra_{d}", lowBound=0)
    prob2 += capacity + extra >= d
    expected_recourse += p * (22 * extra)   # expensive emergency production

prob2 += 15 * capacity + expected_recourse

prob2.solve(pulp.PULP_CBC_CMD(msg=False))

print("=== Two-Stage Capacity Planning Solution ===")
print(f"Optimal capacity to build : {pulp.value(capacity):.1f} units")
print(f"Expected total cost       : ${pulp.value(prob2.objective):.2f}")

## Exercises

1. **Convergence Study**: Repeat the SAA model with 100, 500, 2000, and 5000 scenarios. Plot how the optimal order quantity changes with the number of scenarios.

2. **Sensitivity Analysis**: Vary the ratio `c_u / c_o` from 1 to 20 and observe its effect on the optimal order quantity.

3. **Value of Stochastic Solution (VSS)**: Solve a deterministic newsvendor using only the mean demand, then evaluate its true expected cost on the scenarios. Compare with the SAA solution.

4. **Multi-product Newsvendor**: Extend the model to two products with correlated demand.

5. **Chance-Constrained Version**: Reformulate the problem to ensure the probability of stockout is at most 5%.